# Bab 5 — Missing Values Analysis

Notebook pendamping Bab 5 Level 2. Nilai kosong dibuat secara sengaja (simulasi) karena data asli tidak punya.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

In [ ]:
import os
from pathlib import Path

# Pindah ke folder utama proyek agar "data/mall_customers.csv" ketemu.
if not Path("data").exists():
    os.chdir("../..")

## Langkah 1: Siapkan alat dan muat data

In [ ]:
import numpy as np
import pandas as pd

DATA_PATH = "data/mall_customers.csv"
df = pd.read_csv(DATA_PATH)

## Langkah 2: Periksa data asli

In [ ]:
print("Nilai kosong pada data asli:", df.isna().sum().sum())

## Langkah 3: Buat salinan dengan nilai kosong buatan

In [ ]:
rng = np.random.default_rng(42)  # seed tetap, hasil selalu sama

df_kosong = df.copy()  # salinan, data asli tidak berubah
for kolom in ["Age", "Annual Income (k$)"]:
    baris_terpilih = rng.choice(df_kosong.index, size=10, replace=False)
    df_kosong.loc[baris_terpilih, kolom] = np.nan

print("Nilai kosong pada salinan:")
print(df_kosong.isna().sum())

## Langkah 4: Temukan nilai kosong

In [ ]:
print("Jumlah kosong per kolom:")
print(df_kosong.isna().sum())

print("\nPersentase kosong per kolom:")
print((df_kosong.isna().mean() * 100).round(1))

print("\nJumlah baris yang punya minimal satu nilai kosong:",
      df_kosong.isna().any(axis=1).sum())

print("\nContoh baris yang kosong:")
print(df_kosong[df_kosong.isna().any(axis=1)].head())

## Langkah 5: Strategi 1: buang baris yang kosong

In [ ]:
df_buang = df_kosong.dropna()

print("Ukuran sebelum:", df_kosong.shape)
print("Ukuran sesudah:", df_buang.shape)

## Langkah 6: Strategi 2: isi dengan median

In [ ]:
df_isi = df_kosong.copy()
for kolom in ["Age", "Annual Income (k$)"]:
    median = df_isi[kolom].median()
    df_isi[kolom] = df_isi[kolom].fillna(median)
    print(f"Median {kolom}: {median}")

print("\nNilai kosong tersisa:", df_isi.isna().sum().sum())
print("Ukuran:", df_isi.shape)

## Langkah 7: Bandingkan hasil dengan data asli

In [ ]:
kolom_cek = ["Age", "Annual Income (k$)"]
tabel = pd.DataFrame(
    {
        "asli": df[kolom_cek].mean(),
        "buang baris": df_buang[kolom_cek].mean(),
        "isi median": df_isi[kolom_cek].mean(),
    }
).round(2)
print("Rata-rata:")
print(tabel)

## Langkah 8: Strategi 2 dengan scikit-learn

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
df_sklearn = df_kosong.copy()
df_sklearn[kolom_cek] = imputer.fit_transform(df_sklearn[kolom_cek])

print("Nilai kosong tersisa:", df_sklearn.isna().sum().sum())
print("Median yang dipakai :", imputer.statistics_)